In [ ]:
import os
import cv2
import numpy as np

TRAIN_DIR='dataset'
IMG_SIZE=(128,128)

def extract_features(image_path): 
    image = cv2.imread(image_path) 
    if image is None: 
     return None 
     image = cv2.resize(image, IMG_SIZE) 

    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)     
    hist_h = cv2.calcHist([hsv_image], [0], None, [8], [0, 180])     
    hist_s = cv2.calcHist([hsv_image], [1], None, [8], [0, 256])     
    hist_v = cv2.calcHist([hsv_image], [2], None, [8], [0, 256])     
    color_features = np.concatenate([hist_h, hist_s, hist_v]).flatten()     
    cv2.normalize(color_features, color_features) 

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)     
    moments = cv2.moments(gray)     
    hu_moments = cv2.HuMoments(moments).flatten() 
 
    return np.hstack([color_features, hu_moments]) 
 
def main():
    X_train = []     
    y_train = []  
   
    categories = ['cats', 'dogs'] 

    print(" ... جاري معالجة الصور واستخراج الميزات")     
    for label, category in enumerate(categories):         
        folder_path = os.path.join(TRAIN_DIR, category)         
        if not os.path.exists(folder_path): 
            continue 

        for filename in os.listdir(folder_path):             
            img_path = os.path.join(folder_path, filename)             
            feat = extract_features(img_path)             
            if feat is not None:                 
                X_train.append(feat)                 
                y_train.append(label)  

    X_train = np.array(X_train)     
    y_train = np.array(y_train)   

    np.savez('processed_data.npz', X_train=X_train, y_train=y_train)     
    print(f" processed_data.npzصورة بنجاح في ملف  {len(X_train)} تم حفظ ميزات ") 
 
if __name__ == '__main__':     
    main() 
 
    

 ... جاري معالجة الصور واستخراج الميزات
 processed_data.npzصورة بنجاح في ملف  28 تم حفظ ميزات 


In [25]:

import numpy as np 
from sklearn.svm import SVC 
from sklearn.metrics import accuracy_score
import pickle 


def main():     
    print(" ... جاري تحميل الميزات المستخرجة ")    
    data = np.load('processed_data.npz')     
    X_train = data['X_train']     
    y_train = data['y_train'] 
 
 
    print("  SVM  جاري تدريب نموذج ...")     
    model = SVC(kernel='rbf', C=10.0, gamma='scale')     
    model.fit(X_train, y_train) 
    print("___________________________________________")
    train_preds=model.predict(X_train)
    train_acc=accuracy_score(y_train,train_preds)*100
    print(f" نسبة نجاح النموذج في التدريب \n(Training Accuracy):{train_acc:.1f}%")
    print("___________________________________________")

    with open('model.pkl', 'wb') as f:         
        pickle.dump(model, f)              
    print("model.pkl تم تدريب النموذج وحفظه بنجاح في ملف ") 
 
if __name__ == '__main__':     
    main() 

 ... جاري تحميل الميزات المستخرجة 
  SVM  جاري تدريب نموذج ...
___________________________________________
 نسبة نجاح النموذج في التدريب 
(Training Accuracy):100.0%
___________________________________________
model.pkl تم تدريب النموذج وحفظه بنجاح في ملف 


In [28]:
import os 
import cv2 
import numpy as np 
import pandas as pd 
import pickle 
from sklearn.metrics import accuracy_score

 
TEST_DIR = 'test' 
IMG_SIZE = (128, 128) 
 
def extract_features(image_path):     
    image = cv2.imread(image_path)     
    if image is None:         
        return None     
    image = cv2.resize(image, IMG_SIZE)

    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)     
    hist_h = cv2.calcHist([hsv_image], [0], None, [8], [0, 180])     
    hist_s = cv2.calcHist([hsv_image], [1], None, [8], [0, 256])     
    hist_v = cv2.calcHist([hsv_image], [2], None, [8], [0, 256])     
    color_features = np.concatenate([hist_h, hist_s, hist_v]).flatten()     
    cv2.normalize(color_features, color_features) 
 
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)     
    moments = cv2.moments(gray)     
    hu_moments = cv2.HuMoments(moments).flatten() 
 
    return np.hstack([color_features, hu_moments]) 


def main():     
    if not os.path.exists('model.pkl'):         
        print("train.ipynb  لم يتم العثور على النموذج المدرب! يرجى تشغيل أوﻻً.")         
        return 
 
    with open('model.pkl', 'rb') as f:         
        model = pickle.load(f) 
 
    X_test = []     
    file_names = [] 


    print("  جاري فحص صور اﻻختبار في مجلد test...")     
    if os.path.exists(TEST_DIR):         
        for filename in os.listdir(TEST_DIR):             
            img_path = os.path.join(TEST_DIR, filename)             
            feat = extract_features(img_path)             
            if feat is not None:                 
                X_test.append(feat)                 
                file_names.append(filename)


    if len(X_test) > 0:
        
        predictions = model.predict(X_test)
        class_map = {0: 'Cat', 1: 'Dog'}
        pred_labels = [class_map[p] for p in predictions]
        
        actual_labels = ['Cat' if 'cat' in name.lower() else 'Dog' for name in file_names]
        actual_binary = [0 if label == 'Cat' else 1 for label in actual_labels]
        
        results = pd.DataFrame({
            '-اسم الصورة-': file_names,
            '-التصنيف الحقيقي (Actual)-': actual_labels,
            '-التصنيف التلقائي (Predicted)-': pred_labels,
            '-الحالة': [' True' if a == p else ' False' for a, p in zip(actual_labels, pred_labels)]
        })
        
        print("\n---  جدول نتائج واختبار الصور ---")
        print(results.to_string(index=False))
        
        acc = accuracy_score(actual_binary, predictions) * 100
        print(f"\ نسبة نجاح النموذج (Accuracy): {acc:.1f}%")


    else:         
        print(" test! لم يتم العثور على صور في مجلد ") 
 
if __name__ == '__main__':     
    main() 

  جاري فحص صور اﻻختبار في مجلد test...

---  جدول نتائج واختبار الصور ---
-اسم الصورة- -التصنيف الحقيقي (Actual)- -التصنيف التلقائي (Predicted)- -الحالة
    cat.jpeg                        Cat                            Cat    True
    cat2.jpg                        Cat                            Cat    True
    cat3.jpg                        Cat                            Cat    True
    cat4.jpg                        Cat                            Cat    True
    cat5.jpg                        Cat                            Cat    True
    dog.jpeg                        Dog                            Dog    True
    dog1.jpg                        Dog                            Dog    True
    dog2.jpg                        Dog                            Cat   False
\ نسبة نجاح النموذج (Accuracy): 87.5%
